In [1]:
import seaborn as sns
import plotly.express as px
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.backends.backend_pdf import PdfPages
from lifelines import CoxPHFitter  # for Cox survival models
from pymer4.models import Lmer
import os
import geopandas as gpd
from rra_climate_health.data import ClimateMalnutritionData, DEFAULT_ROOT
from pathlib import Path

In [23]:
DATA_PATH_4_28 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/training_data/2026_04_28.01/data_cumulative_bins.parquet"
DATA_PATH_10_07 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/training_data/2026_10_07.01/data.parquet"

PLOT_PATH = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/plots/"

In [3]:
df_428 = pd.read_parquet(DATA_PATH_4_28)
df_1007 = pd.read_parquet(DATA_PATH_10_07)

In [4]:
df_428 = df_428[df_428["int_birth_year_diff_months"] <= 120]

In [22]:
def plot_heat_map_person_time(
    data: pd.DataFrame,
    outfile: str,
    title: str,
    bin_cols: list,
    format: str = ".2f",
    multiply_by: int = 1,
    vmin: float = None,
    vmax: float = None,
    show_colorbar: bool = True,
    x_bins: list = None,
    y_bins: list = None,
):
    heatmap_df = data.copy()
    for col in bin_cols:
        new_bin_col = f"{col}_bin"
        # heatmap_df[new_bin_col] = pd.qcut(
        #     heatmap_df[col], 10, retbins=False, duplicates="drop"
        # )
        if x_bins is not None:
            heatmap_df[f"{col}_bin"] = pd.cut(
                heatmap_df[col], bins=x_bins, include_lowest=True, right=False
            )
        else:
            heatmap_df[f"{col}_bin"] = pd.cut(
                heatmap_df[col], bins=10, include_lowest=True
            )
    if y_bins is None:
        heatmap_df["consumption_pd_bin"], ldi_bins = pd.qcut(
            heatmap_df.consumption_pd, 10, retbins=True, duplicates="drop"
        )
    else:
        heatmap_df["consumption_pd_bin"] = pd.cut(
            heatmap_df.consumption_pd,
            bins=y_bins,
            include_lowest=True,
            right=False,
        )

    for col in bin_cols:
        figsize = (6, 6) if not show_colorbar else (7, 6)
        plt.figure(figsize=figsize)

        heatmap_data = (
            heatmap_df.groupby(["consumption_pd_bin", f"{col}_bin"])[
                "model_predictions"
            ]
            .sum()
            .unstack()
            / heatmap_df.groupby(["consumption_pd_bin", f"{col}_bin"])["age_month"]
            .sum()
            .unstack()
        )
        heatmap_data *= multiply_by

        if vmin is None:
            vmin = heatmap_data.min().min()
        if vmax is None:
            vmax = heatmap_data.max().max()
        colorbin_interval = (vmax - vmin) / 10
        boundaries = np.arange(vmin, vmax + colorbin_interval, colorbin_interval)
        cmap = plt.get_cmap("RdYlBu_r", len(boundaries) - 1)
        norm = mcolors.BoundaryNorm(boundaries, cmap.N, clip=True)

        x_tick_vals = heatmap_df.groupby([new_bin_col])[col].min().values.tolist() + [
            int(heatmap_df[col].max())
        ]
        y_tick_vals = heatmap_df.groupby(
            ["consumption_pd_bin"]
        ).consumption_pd.min().values.tolist() + [heatmap_df.consumption_pd.max()]
        x_ticks = range(len(x_tick_vals) + 1)
        y_ticks = range(len(y_tick_vals) + 1)
        x_labs = [f"{x_tick_vals[i]:.1f}" for i in range(len(x_tick_vals))]
        y_labs = [f"{y_tick_vals[i]:.1f}" for i in range(len(y_tick_vals))]

        ax = sns.heatmap(
            heatmap_data,
            annot=True,
            fmt=format,
            cmap=cmap,
            norm=norm,
            vmin=vmin,
            vmax=vmax,
            annot_kws={"size": 10, "weight": "regular"},
            cbar=show_colorbar,
        )
        if show_colorbar:
            cbar = ax.collections[0].colorbar
            cbar.ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f"))
        # ax.set_xticks(x_ticks)
        n_rows, n_cols = heatmap_data.shape
        ax.set_xticks(np.arange(n_cols + 1))
        ax.set_yticks(np.arange(n_rows + 1))
        ax.set_xticklabels(x_labs, rotation=45, fontsize=10)
        # ax.set_yticks(y_ticks)
        ax.set_yticklabels(y_labs, rotation=0, fontsize=10)
        ax.set_xlabel("Days over 30°C", fontsize=13)
        ax.set_ylabel("Daily consumption", fontsize=13)
        ax.set_title(title, fontsize=18)

        plt.tight_layout()
        plt.savefig(
            os.path.join(PLOT_PATH, f"{outfile}_{col}.png"), bbox_inches="tight"
        )
        plt.close()
        plt.show()

In [7]:
# Formatting

# Add age bin
for v in [
    "age_1_m",
    "age_3_m",
    "age_6_m",
    "age_12_m",
    "age_24_m",
    "age_36_m",
    "age_48_m",
    "age_60_m",
]:
    df_428.loc[df_428[v] == 1, "age_group"] = v
    df_1007.loc[df_1007[v] == 1, "age_group"] = v

age_group_durations = {
    "age_1_m": 1,
    "age_3_m": 2,
    "age_6_m": 3,
    "age_12_m": 6,
    "age_24_m": 12,
    "age_36_m": 12,
    "age_48_m": 12,
    "age_60_m": 12,
}

custom_x_bins = [0, 0.1, 2, 4, 9, 31]

custom_y_bins = [
    0,
    0.784781,
    1.180789,
    1.541445,
    1.950251,
    2.465952,
    3.103463,
    4.003564,
    5.541124,
    9.413681,
    112.879922,
]

df_428.rename(columns={"child_mortality": "model_predictions"}, inplace=True)
df_1007.rename(columns={"child_mortality": "model_predictions"}, inplace=True)
df_428.drop(columns=["age_month"], inplace=True)
df_428["age_month"] = df_428["age_group"].map(age_group_durations)
df_1007.drop(columns=["age_month"], inplace=True)
df_1007["age_month"] = df_1007["age_group"].map(age_group_durations)

In [8]:
if "consumption_pd" in df_428.columns:
    df_428.drop(columns="consumption_pd", inplace=True)
if "consumption_pd" in df_1007.columns:
    df_1007.drop(columns="consumption_pd", inplace=True)

In [9]:
df_428.rename(columns={"consumption_pd_cumul": "consumption_pd"}, inplace=True)
df_1007.rename(columns={"consumption_pd_cumul": "consumption_pd"}, inplace=True)

In [ ]:
# Drop the follow-up rows for children who died ###############################
#
# This file keeps a child in the data AFTER they die and leaves them flagged in
# every remaining interval, so `model_predictions` counts intervals-spent-dead
# rather than deaths. It does not do this uniformly: 83.7% of dying children are
# already truncated at death, 16.3% are carried forward, and which happens
# depends on the SURVEY - the carried-forward share runs from 1.5% to 83% across
# NIDs. That makes the raw sum weight a few surveys far more heavily than the
# rest.
#
# Below keeps every row up to AND INCLUDING the first interval a child is
# flagged in, and drops anything after. Children who never die keep all their
# rows. The result counts each death once, which is what the 2026_10_07.01
# extract does natively.
_ORDER = [
    "age_1_m",
    "age_3_m",
    "age_6_m",
    "age_12_m",
    "age_24_m",
    "age_36_m",
    "age_48_m",
    "age_60_m",
]

# age_month was overwritten with interval DURATIONS in the formatting cell
# above (1, 2, 3, 6, 12, 12, 12, 12), so the last four intervals tie and it can
# no longer be used to order a child's rows. Order on the age_group sequence.
_k = df_428["age_group"].map({a: i for i, a in enumerate(_ORDER)})
if _k.isna().any():
    raise ValueError("age_group values outside the expected eight intervals")

# first flagged interval per child; NaN for children who never die
_first_death = (
    _k.where(df_428["model_predictions"] == 1).groupby(df_428["indv_id"]).min()
)
_cut = df_428["indv_id"].map(_first_death).fillna(np.inf)

df_428_trunc = df_428[_k <= _cut].copy()

_n_child = df_428["indv_id"].nunique()
_ever = int((df_428.groupby("indv_id")["model_predictions"].max() == 1).sum())
print(
    "df_428        rows {:>12,}  children {:>10,}  flags {:>9,}".format(
        len(df_428), _n_child, int(df_428["model_predictions"].sum())
    )
)
print(
    "df_428_trunc  rows {:>12,}  children {:>10,}  flags {:>9,}".format(
        len(df_428_trunc),
        df_428_trunc["indv_id"].nunique(),
        int(df_428_trunc["model_predictions"].sum()),
    )
)
print(
    "children that ever die: {:,}  (flags in the truncated frame should match)".format(
        _ever
    )
)
print("rows removed: {:,}".format(len(df_428) - len(df_428_trunc)))

# the truncation must not drop a child entirely, and must leave exactly one flag
# per dying child
assert df_428_trunc["indv_id"].nunique() == _n_child, "a child lost all of its rows"
_s = df_428_trunc.groupby("indv_id")["model_predictions"].sum()
assert _s.max() <= 1, "a child still has more than one flag"
assert int(_s.sum()) == _ever, "flag total does not match the number of dying children"
print("checks passed: one flag per dying child, no child dropped")

In [27]:
if "days_over_30C" in df_428.columns:
    df_428.drop(columns="days_over_30C", inplace=True)
if "days_over_30C" in df_428_trunc.columns:
    df_428_trunc.drop(columns="days_over_30C", inplace=True)
if "days_over_30C" in df_1007.columns:
    df_1007.drop(columns="days_over_30C", inplace=True)

df_428.rename(columns={"days_over_30C_monthly_cumul": "days_over_30C"}, inplace=True)
df_1007.rename(columns={"days_over_30C_monthly_cumul": "days_over_30C"}, inplace=True)
df_428_trunc.rename(
    columns={"days_over_30C_monthly_cumul": "days_over_30C"}, inplace=True
)

In [28]:
vmin = 1.0
vmax = 7.0
multiply_by_val = 1000

columns_to_bin = [
    "days_over_30C",
]

df_tmp = df_428.copy()
df_tmp.rename(columns={"child_mortality": "model_predictions"}, inplace=True)

plot_heat_map_person_time(
    data=df_tmp,
    outfile="child_mortality_heatmap_raw_data_4_28_2026",
    title="Raw data from 4-28-2026",
    bin_cols=columns_to_bin,
    format=".2f",
    multiply_by=multiply_by_val,
    vmin=vmin,
    vmax=vmax,
    show_colorbar=False,
    x_bins=custom_x_bins,
    y_bins=custom_y_bins,
)  # 0.5 to 2.5

/tmp/ipykernel_2832982/4173715548.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  heatmap_df.groupby(["consumption_pd_bin", f"{col}_bin"])[
/tmp/ipykernel_2832982/4173715548.py:50: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  / heatmap_df.groupby(["consumption_pd_bin", f"{col}_bin"])["age_month"]
/tmp/ipykernel_2832982/4173715548.py:65: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  x_tick_vals = heatmap_df.groupby(

In [29]:
vmin = 1.0
vmax = 7.0
multiply_by_val = 1000

columns_to_bin = [
    "days_over_30C",
]

df_tmp = df_428_trunc.copy()
df_tmp.rename(columns={"child_mortality": "model_predictions"}, inplace=True)

plot_heat_map_person_time(
    data=df_tmp,
    outfile="child_mortality_heatmap_raw_data_truncated_4_28_2026",
    title="Raw data from 4-28-2026, truncated",
    bin_cols=columns_to_bin,
    format=".2f",
    multiply_by=multiply_by_val,
    vmin=vmin,
    vmax=vmax,
    show_colorbar=False,
    x_bins=custom_x_bins,
    y_bins=custom_y_bins,
)  # 0.5 to 2.5

/tmp/ipykernel_2832982/4173715548.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  heatmap_df.groupby(["consumption_pd_bin", f"{col}_bin"])[
/tmp/ipykernel_2832982/4173715548.py:50: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  / heatmap_df.groupby(["consumption_pd_bin", f"{col}_bin"])["age_month"]
/tmp/ipykernel_2832982/4173715548.py:65: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  x_tick_vals = heatmap_df.groupby(

In [30]:
vmin = 1.0
vmax = 7.0
multiply_by_val = 1000

columns_to_bin = [
    "days_over_30C",
]

df_tmp = df_1007.copy()
df_tmp.rename(columns={"child_mortality": "model_predictions"}, inplace=True)

plot_heat_map_person_time(
    data=df_tmp,
    outfile="child_mortality_heatmap_raw_data_10_07_2026",
    title="Raw data from 10-07-2026",
    bin_cols=columns_to_bin,
    format=".2f",
    multiply_by=multiply_by_val,
    vmin=vmin,
    vmax=vmax,
    show_colorbar=False,
    x_bins=custom_x_bins,
    y_bins=custom_y_bins,
)  # 0.5 to 2.5

/tmp/ipykernel_2832982/4173715548.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  heatmap_df.groupby(["consumption_pd_bin", f"{col}_bin"])[
/tmp/ipykernel_2832982/4173715548.py:50: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  / heatmap_df.groupby(["consumption_pd_bin", f"{col}_bin"])["age_month"]
/tmp/ipykernel_2832982/4173715548.py:65: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  x_tick_vals = heatmap_df.groupby(

In [11]:
print(df_428["age_group"].value_counts())
print(df_1007["age_group"].value_counts())

age_group
age_1_m     3149184
age_3_m     3054073
age_6_m     2990045
age_12_m    2902448
age_24_m    2737795
age_36_m    2436987
age_48_m    2140778
age_60_m    1840928
Name: count, dtype: int64
age_group
age_1_m     3588182
age_3_m     3466012
age_6_m     3376269
age_12_m    3255218
age_24_m    3033530
age_36_m    2639904
age_48_m    2264674
age_60_m    1891400
Name: count, dtype: int64


df_428        rows   21,252,238  children  3,149,184  flags   435,776
df_428_trunc  rows   21,066,555  children  3,149,184  flags   250,093
children that ever die: 250,093  (flags in the truncated frame should match)
rows removed: 185,683
checks passed: one flag per dying child, no child dropped


/tmp/ipykernel_2832982/4173715548.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  heatmap_df.groupby(["consumption_pd_bin", f"{col}_bin"])[
/tmp/ipykernel_2832982/4173715548.py:50: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  / heatmap_df.groupby(["consumption_pd_bin", f"{col}_bin"])["age_month"]
/tmp/ipykernel_2832982/4173715548.py:65: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  x_tick_vals = heatmap_df.groupby(

In [13]:
df_428["model_predictions"].value_counts()

model_predictions
0    20816462
1      435776
Name: count, dtype: int64

In [15]:
df_1007["model_predictions"].value_counts()

model_predictions
0    23265086
1      250103
Name: count, dtype: int64

In [16]:
ex_id1 = df_428[df_428["model_predictions"] == 1]["indv_id"].iloc[0]
print(df_428[df_428["indv_id"] == ex_id1])

   ihme_loc_id  geospatial_id  psu strata  line_id  hh_id       lat      long  \
38         MAR            207  207     22        4      7  33.90823 -5.340197   

    lbd_admin2_id        indv_id  ...  mean_temperature_monthly_constant  \
38          75908  20361_207_7_4  ...                          10.976774   

    total_precipitation_monthly_constant  q75_monthly_constant  \
38                              9.196774                   0.0   

    q80_monthly_constant  q85_monthly_constant q90_monthly_constant  \
38                   0.0                   0.0                  0.0   

   q95_monthly_constant  lower_bound  upper_bound  age_month  
38                  0.0            0            1          1  

[1 rows x 91 columns]


In [18]:
df_428[df_428["indv_id"] == ex_id1]

,ihme_loc_id,geospatial_id,psu,strata,line_id,hh_id,lat,long,lbd_admin2_id,indv_id,...,mean_temperature_monthly_constant,total_precipitation_monthly_constant,q75_monthly_constant,q80_monthly_constant,q85_monthly_constant,q90_monthly_constant,q95_monthly_constant,lower_bound,upper_bound,age_month
38,MAR,207,207,22,4,7,33.90823,-5.340197,75908,20361_207_7_4,...,10.976774,9.196774,0.0,0.0,0.0,0.0,0.0,0,1,1


In [19]:
ex_id1

'20361_207_7_4'